# 10 — Model Merging: Combine Adapters Without Extra Training

> Part of the [SLM Fine-Tuning Tutorial](https://github.com/Kumarjit-Pathak/finetuning-slm)

Model merging combines multiple fine-tuned LoRA adapters into a single multi-task model — **zero additional training required**. Introduced in model soups and refined by TIES/DARE/SLERP, this technique is increasingly used in open-source model development.

| Method | Training Required | Multi-task Quality | Key Idea |
|---|---|---|---|
| Train from scratch | Heavy (full multi-task data) | Best possible | Joint training on all tasks |
| Sequential fine-tuning | Medium (each task separately) | Catastrophic forgetting | Overwrites previous task |
| **Linear Interpolation** | ❌ None | Good (balanced) | λ·W_A + (1-λ)·W_B |
| **TIES** | ❌ None | Better | Trim + Elect Sign + Merge |
| **DARE** | ❌ None | Better | Drop & Rescale deltas |
| **SLERP** | ❌ None | Smooth | Spherical interpolation |

In [1]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q",
                    "torch", "transformers", "peft", "trl",
                    "datasets", "accelerate", "matplotlib"], check=True)
    print("Colab packages installed.")
else:
    print("Local environment — skipping pip install.")

Local environment — skipping pip install.


## Why Model Merging?

Training a multi-task model traditionally requires all task data simultaneously and is expensive. Model merging offers an alternative:

1. **Fine-tune separate specialist adapters** (cheap, in parallel, on different datasets)
2. **Merge their weight deltas** (instant, no GPU needed, fully reversible)
3. **Get a generalist** that retains knowledge from all tasks

### The Core Idea: Weight Deltas

Every fine-tuned model can be decomposed as:

```
W_finetuned = W_base + ΔW_task
```

Where `ΔW_task` is the *task vector* — a small perturbation capturing what the model learned. Merging is arithmetic on these task vectors:

```
ΔW_merged = λ·ΔW_A + (1-λ)·ΔW_B          ← Linear Interpolation
ΔW_merged = TIES(ΔW_A, ΔW_B)              ← Resolve sign conflicts first
ΔW_merged = DARE(ΔW_A, ΔW_B, drop=0.3)   ← Random drop + rescale
ΔW_merged = SLERP(ΔW_A, ΔW_B, t=0.5)    ← Interpolate on hypersphere
```

### This Notebook

- **Task A**: Q&A about ML concepts → fine-tune Adapter A
- **Task B**: Text summarization → fine-tune Adapter B
- Merge them with 4 methods, evaluate on **both tasks** simultaneously
- 🔴 **Failure demo**: merging incompatible adapters → quality collapses on both

In [2]:
import os, gc, copy, time, tempfile
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# ── Offline mode ──
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.manual_seed(42)

# ── Device ──
device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available()          else "cpu"
)
print(f"Device: {device}")

# ── Model selection (cached only) ──
CACHE = os.path.expanduser("~/.cache/huggingface/hub")
MODEL = None
for candidate in ["HuggingFaceTB/SmolLM2-360M-Instruct",
                  "HuggingFaceTB/SmolLM2-135M-Instruct"]:
    slug = "models--" + candidate.replace("/", "--")
    if os.path.isdir(os.path.join(CACHE, slug)):
        MODEL = candidate
        break
assert MODEL, "No cached SmolLM2 model found — run notebook 00 first."
print(f"Model : {MODEL}")

# ── Helpers ──
def load_base():
    """Load a fresh copy of the base model (no LoRA)."""
    m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)
    return m.to(device)

def free_memory(*args):
    for obj in args:
        del obj
    gc.collect()
    if device.type == "mps":
        torch.mps.empty_cache()

# ── Shared LoRA config ──
LORA_CFG = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM,
)

# ── Load tokenizer ──
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer vocab: {tokenizer.vocab_size:,}")
print("Setup complete ✓")

Device: mps
Model : HuggingFaceTB/SmolLM2-135M-Instruct
Tokenizer vocab: 49,152
Setup complete ✓


## Two Specialist Adapters

We train two LoRA adapters on **completely different tasks** to ensure they develop distinct, non-overlapping specializations:

| | Task A — Q&A | Task B — Summarization |
|---|---|---|
| **Format** | `Q: [question]\nA:` → detailed answer | `Summarize: [long text]\n\nSummary:` → brief summary |
| **Skill** | Recall & explain ML concepts | Compress information, extract essence |
| **Training** | 16 pairs × 4 repeats × 5 epochs = 320 steps | 16 pairs × 4 repeats × 5 epochs = 320 steps |
| **Expected outcome** | Low QA loss, high Sum loss | High QA loss, low Sum loss |

After training, we extract the **task vectors** (weight deltas) from each adapter and combine them using various merging strategies — without any additional training.

In [4]:
# ── Task A: Q&A Dataset  (concise, notation-consistent answers) ──
QA_TRAIN = [
    {"input": "Q: What is gradient descent?\nA:", "target": " Optimizer that updates weights by w ← w − η·∇L, iteratively minimizing loss."},
    {"input": "Q: What is a transformer?\nA:", "target": " Neural architecture using self-attention: Attention(Q,K,V) = softmax(QK^T/√d_k)·V."},
    {"input": "Q: What is self-attention?\nA:", "target": " Each token attends to all others via Q·K^T/√d_k softmax scores, then weighted V sum."},
    {"input": "Q: What is LoRA?\nA:", "target": " Low-rank adaptation: ΔW = B·A where rank r ≪ d, reducing parameters by d·k/(r·(d+k))."},
    {"input": "Q: What is quantization?\nA:", "target": " Maps float32 weights to int8: q = round(w / scale), cutting model size ~4×."},
    {"input": "Q: What is fine-tuning?\nA:", "target": " Continues training pre-trained weights on task data with low η, adapting representations."},
    {"input": "Q: What is overfitting?\nA:", "target": " train_loss ≪ val_loss; model memorizes training data, failing to generalize."},
    {"input": "Q: What is dropout?\nA:", "target": " Randomly zeros p fraction of activations; survivors rescaled by 1/(1−p) during training."},
    {"input": "Q: What is batch normalization?\nA:", "target": " Normalizes each feature: z = (x − μ_B) / σ_B, then scales with learned γ, β."},
    {"input": "Q: What is a residual connection?\nA:", "target": " Skip-connection: y = F(x) + x, enabling gradient flow via ∂L/∂x = ∂L/∂y · (1 + ∂F/∂x)."},
    {"input": "Q: What is tokenization?\nA:", "target": " Splits text into subwords via BPE: merge most-frequent byte pairs until vocab size reached."},
    {"input": "Q: What is RLHF?\nA:", "target": " RL with human feedback: train reward r(x,y), then maximize E[r] − β·KL(π‖π_ref) via PPO."},
    {"input": "Q: What is knowledge distillation?\nA:", "target": " Loss = α·CE(y_hard) + (1−α)·T²·KL(student‖teacher_soft), training small model on teacher logits."},
    {"input": "Q: What is backpropagation?\nA:", "target": " Chain rule applied backward: ∂L/∂W_i = ∂L/∂a_i · ∂a_i/∂W_i, layer by layer."},
    {"input": "Q: What is DPO?\nA:", "target": " Alignment loss: −log σ(β·log π(y_w|x)/π_ref(y_w|x) − β·log π(y_l|x)/π_ref(y_l|x))."},
    {"input": "Q: What is perplexity?\nA:", "target": " PPL = exp(−(1/N)·Σ log p(t_i)), average branching factor of the model's predictions."},
]

QA_VAL = [
    {"input": "Q: What is the attention mechanism?\nA:", "target": " Attention(Q,K,V) = softmax(QK^T/√d_k)·V; each token selectively weighs all other tokens."},
    {"input": "Q: What is weight decay?\nA:", "target": " L2 regularization: loss += λ·Σ‖w‖², shrinking weights toward 0, reducing overfitting."},
    {"input": "Q: What is a learning rate scheduler?\nA:", "target": " Adjusts η over training, e.g., linear warmup then cosine decay to η_min."},
    {"input": "Q: What is layer normalization?\nA:", "target": " Normalizes per-sample across features: z = (x − μ) / (σ + ε), with learned scale γ, β."},
]

# ── Task B: Summarization Dataset  (concise, formula-free summaries) ──
SUM_TRAIN = [
    {"input": "Summarize: Transformers process entire sequences in parallel using self-attention, where each token attends to every other token simultaneously via dot-product scores. This parallelism enables efficient GPU training at scale.\n\nSummary:", "target": " Transformers enable parallel sequence processing via self-attention, unlocking efficient large-scale training."},
    {"input": "Summarize: LoRA reduces trainable parameters by learning two small matrices A and B instead of updating the full weight matrix. The rank r controls the trade-off between expressiveness and parameter count.\n\nSummary:", "target": " LoRA learns low-rank weight updates, dramatically cutting trainable parameters while preserving performance."},
    {"input": "Summarize: Quantization converts float32 weights to lower-bit integers, reducing memory 4-8×. Post-training quantization applies without retraining, making large models accessible on consumer hardware.\n\nSummary:", "target": " Quantization reduces memory 4-8× by lowering weight precision, enabling large models on consumer hardware."},
    {"input": "Summarize: Knowledge distillation trains a small student to match a large teacher's soft output distributions. Soft labels carry richer information than hard labels, enabling the student to inherit the teacher's uncertainty.\n\nSummary:", "target": " Distillation transfers a teacher's soft-label knowledge to a smaller student, exceeding hard-label training."},
    {"input": "Summarize: RLHF involves three stages: supervised fine-tuning on demonstrations, training a reward model on human preference comparisons, and optimizing the policy with PPO under a KL constraint.\n\nSummary:", "target": " RLHF aligns models through SFT, reward modeling on preferences, and PPO-based policy optimization."},
    {"input": "Summarize: Model merging combines fine-tuned task vectors ΔW_A and ΔW_B without additional training. The merged model inherits multi-task capability via W_base + λ·ΔW_A + (1-λ)·ΔW_B.\n\nSummary:", "target": " Model merging interpolates task vectors to create a multi-task model with zero extra training."},
    {"input": "Summarize: Gradient descent approximates the full gradient with mini-batches (SGD), making optimization feasible for large datasets. Adaptive methods like Adam add per-parameter learning rates via moment estimates.\n\nSummary:", "target": " SGD approximates gradients via mini-batches; Adam adds adaptive per-parameter rates via moments."},
    {"input": "Summarize: Multi-head attention runs H parallel attention heads and concatenates their outputs: MHA(Q,K,V) = Concat(head_1,...,head_H)·W_O. Each head attends to different parts of the representation.\n\nSummary:", "target": " Multi-head attention runs parallel attention heads, each capturing different representational aspects."},
    {"input": "Summarize: DPO eliminates the separate reward model by deriving a closed-form policy objective from preference pairs, yielding a simple binary cross-entropy loss that avoids reward hacking.\n\nSummary:", "target": " DPO converts preference pairs directly into a policy loss, bypassing reward model training."},
    {"input": "Summarize: Prefix tuning prepends learned soft token embeddings to the frozen base model, training only these prefix vectors. As few as 0.1% of parameters need updating for strong generation performance.\n\nSummary:", "target": " Prefix tuning trains only a small set of soft tokens prepended to a frozen model, reaching strong results."},
    {"input": "Summarize: Full fine-tuning updates all parameters and requires optimizer states (Adam needs 3× model size in memory), making it infeasible on consumer hardware for large models.\n\nSummary:", "target": " Full fine-tuning updates all parameters and demands large memory for optimizer states, limiting hardware access."},
    {"input": "Summarize: QLoRA uses 4-bit NF4 quantization for the base model with LoRA adapters trained in bfloat16, cutting GPU memory ~4× versus standard LoRA and enabling fine-tuning of 70B models on one consumer GPU.\n\nSummary:", "target": " QLoRA quantizes the base to 4-bit and trains LoRA in bf16, enabling 70B model fine-tuning on one GPU."},
    {"input": "Summarize: Residual connections (y = F(x) + x) add an identity shortcut that allows gradients to bypass blocks, solving the vanishing gradient problem and enabling training of very deep networks.\n\nSummary:", "target": " Residual connections bypass blocks with identity shortcuts, enabling gradient flow in deep networks."},
    {"input": "Summarize: LayerNorm normalizes per-sample across feature dimensions, independent of batch size. This makes it suitable for variable-length sequences and small batches, unlike BatchNorm.\n\nSummary:", "target": " LayerNorm normalizes per-sample across features, making it ideal for variable-length sequences."},
    {"input": "Summarize: Dropout zeros p fraction of activations during training and rescales survivors by 1/(1-p). At inference all neurons are active. This acts as an implicit ensemble of exponentially many models.\n\nSummary:", "target": " Dropout randomly zeros activations during training, acting as an implicit ensemble that prevents overfitting."},
    {"input": "Summarize: BPE tokenization starts with characters and iteratively merges the most frequent adjacent pairs. This handles rare words through subword splitting while keeping common words as single tokens.\n\nSummary:", "target": " BPE builds vocabulary by merging frequent character pairs, handling rare words via subword splitting."},
]

SUM_VAL = [
    {"input": "Summarize: Flash Attention tiles the N×N attention matrix computation to stay in SRAM, achieving O(N) memory instead of O(N²) while producing identical results, with 2-4× speedup on modern GPUs.\n\nSummary:", "target": " Flash Attention tiles attention to SRAM, reducing memory to O(N) with 2-4× speedup over standard attention."},
    {"input": "Summarize: Scaling laws show language model performance improves as a power law with compute, data, and model size over many orders of magnitude, motivating ever-larger model training.\n\nSummary:", "target": " Scaling laws show model quality improves as power laws with compute, data, and size, motivating large training runs."},
    {"input": "Summarize: Mixture of Experts routes each token to a few selected expert feed-forward networks, allowing total parameter count to scale independently of per-token compute cost.\n\nSummary:", "target": " MoE routes tokens to a few experts, scaling total parameters without proportional compute increase."},
    {"input": "Summarize: Constitutional AI trains models to critique and revise their outputs according to a set of principles, enabling alignment without per-output human feedback at scale.\n\nSummary:", "target": " Constitutional AI uses self-critique against principles to scale alignment beyond per-output human feedback."},
]

# ── Training data (no repetition this time — 16 samples, cleaner learning) ──
QA_TRAIN_EFF  = QA_TRAIN   # 16 samples × 3 epochs = 48 steps
SUM_TRAIN_EFF = SUM_TRAIN  # 16 samples × 3 epochs = 48 steps

print(f"Task A (Q&A):          {len(QA_TRAIN)} train, {len(QA_VAL)} val")
print(f"Task B (Summarization): {len(SUM_TRAIN)} train, {len(SUM_VAL)} val")

# ── Training and evaluation helpers ──
MAX_LEN = 192

def compute_val_loss(model, dataset):
    """Average cross-entropy loss on a dataset (response tokens only)."""
    model.eval()
    total_loss, count = 0.0, 0
    with torch.no_grad():
        for item in dataset:
            full_text = item["input"] + item["target"]
            enc = tokenizer(full_text, return_tensors="pt",
                            max_length=MAX_LEN, truncation=True).to(device)
            prompt_len = tokenizer(item["input"], return_tensors="pt",
                                   max_length=MAX_LEN, truncation=True).input_ids.shape[1]
            labels = enc.input_ids.clone()
            labels[0, :prompt_len] = -100
            if (labels != -100).sum() == 0:
                continue
            out = model(**enc, labels=labels)
            total_loss += out.loss.item()
            count += 1
    return total_loss / count if count > 0 else float("inf")

def train_adapter(base_model, dataset, epochs=3, lr=1e-4):
    """Fine-tune a LoRA adapter on a dataset. Returns (peft_model, losses)."""
    model = get_peft_model(base_model, LORA_CFG)
    model.train()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    losses = []
    for ep in range(epochs):
        ep_loss = 0.0
        for item in dataset:
            full_text = item["input"] + item["target"]
            enc = tokenizer(full_text, return_tensors="pt",
                            max_length=MAX_LEN, truncation=True).to(device)
            prompt_len = tokenizer(item["input"], return_tensors="pt",
                                   max_length=MAX_LEN, truncation=True).input_ids.shape[1]
            labels = enc.input_ids.clone()
            labels[0, :prompt_len] = -100
            out = model(**enc, labels=labels)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            ep_loss += out.loss.item()
        losses.append(ep_loss / len(dataset))
    model.eval()
    return model, losses

from torch.optim import AdamW

# ── Train Adapter A (Q&A) ──
print("\nTraining Adapter A (Q&A) …")
t0 = time.time()
base_a = load_base()
model_a, losses_a = train_adapter(base_a, QA_TRAIN_EFF, epochs=3, lr=1e-4)
merged_a_model = model_a.merge_and_unload()
state_a = {k: v.cpu().clone() for k, v in merged_a_model.state_dict().items()}
free_memory(base_a, model_a, merged_a_model)
print(f"  Adapter A trained in {time.time()-t0:.1f}s  | losses: {[f'{l:.3f}' for l in losses_a]}")

# ── Train Adapter B (Summarization) ──
print("Training Adapter B (Summarization) …")
t0 = time.time()
base_b = load_base()
model_b, losses_b = train_adapter(base_b, SUM_TRAIN_EFF, epochs=3, lr=1e-4)
merged_b_model = model_b.merge_and_unload()
state_b = {k: v.cpu().clone() for k, v in merged_b_model.state_dict().items()}
free_memory(base_b, model_b, merged_b_model)
print(f"  Adapter B trained in {time.time()-t0:.1f}s  | losses: {[f'{l:.3f}' for l in losses_b]}")

# ── Get base model state dict ──
base_ref = load_base()
base_state = {k: v.cpu().clone() for k, v in base_ref.state_dict().items()}

# ── Compute task vectors (weight deltas) ──
def compute_delta(merged_state, base_state, tol=1e-12):
    return {k: (merged_state[k] - base_state[k]).cpu()
            for k in base_state
            if k in merged_state and (merged_state[k] - base_state[k]).abs().max() > tol}

delta_a = compute_delta(state_a, base_state)
delta_b = compute_delta(state_b, base_state)
print(f"\nTask vector A: {len(delta_a)} modified param tensors")
print(f"Task vector B: {len(delta_b)} modified param tensors")

# ── Baseline evaluation ──
def eval_from_delta(delta, label=""):
    """Apply delta to base model and evaluate on both tasks."""
    m = load_base()
    with torch.no_grad():
        for name, param in m.named_parameters():
            if name in delta:
                param.data += delta[name].to(device)
    m.eval()
    qa  = compute_val_loss(m, QA_VAL)
    sur = compute_val_loss(m, SUM_VAL)
    free_memory(m)
    return qa, sur

qa_base,  sum_base  = eval_from_delta({},      "Base")
qa_a,     sum_a     = eval_from_delta(delta_a, "A only")
qa_b,     sum_b     = eval_from_delta(delta_b, "B only")

print(f"\n{'Model':<22} {'QA loss':>9} {'Sum loss':>10}")
print("-" * 43)
print(f"{'Base (no FT)':<22} {qa_base:>9.4f} {sum_base:>10.4f}")
print(f"{'Adapter A (QA)':<22} {qa_a:>9.4f} {sum_a:>10.4f}")
print(f"{'Adapter B (Sum)':<22} {qa_b:>9.4f} {sum_b:>10.4f}")

Task A (Q&A):          16 train, 4 val
Task B (Summarization): 16 train, 4 val

Training Adapter A (Q&A) …


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  Adapter A trained in 4.8s  | losses: ['4.818', '4.681', '4.550']
Training Adapter B (Summarization) …


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  Adapter B trained in 4.8s  | losses: ['2.695', '2.594', '2.479']


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


Task vector A: 60 modified param tensors
Task vector B: 60 modified param tensors


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


Model                    QA loss   Sum loss
-------------------------------------------
Base (no FT)              4.5077     3.1772
Adapter A (QA)            4.1931     3.1695
Adapter B (Sum)           4.4827     2.8832


## Merging Method 1: Linear Interpolation

The simplest merging strategy — weighted average of task vectors:

$$\Delta W_{\text{merged}} = \lambda \cdot \Delta W_A + (1-\lambda) \cdot \Delta W_B$$

$$W_{\text{merged}} = W_{\text{base}} + \Delta W_{\text{merged}}$$

**Intuition**: λ=0 gives pure Adapter B, λ=1 gives pure Adapter A, λ=0.5 gives equal blend.

**Tradeoff**: Linear blending can dilute each task's signal. If the adapters push weights in different directions, averaging partially cancels both. TIES and DARE address this.

We sweep λ ∈ {0.3, 0.5, 0.7} to find the Pareto-optimal blend that minimizes total loss across both tasks.

In [5]:
# ── Linear Interpolation ──
def linear_merge(da, db, lam=0.5):
    """λ·ΔW_A + (1-λ)·ΔW_B for keys present in both."""
    keys = set(da) | set(db)
    merged = {}
    for k in keys:
        if k in da and k in db:
            merged[k] = lam * da[k] + (1 - lam) * db[k]
        elif k in da:
            merged[k] = da[k]
        else:
            merged[k] = db[k]
    return merged

lambdas = [0.0, 0.3, 0.5, 0.7, 1.0]  # 0.0=pure B, 1.0=pure A
linear_results = {}

print(f"{'λ':<8} {'QA loss':>9} {'Sum loss':>10} {'Total':>9}")
print("-" * 40)
for lam in lambdas:
    merged = linear_merge(delta_a, delta_b, lam=lam)
    qa, sur = eval_from_delta(merged, f"λ={lam}")
    linear_results[lam] = (qa, sur)
    print(f"λ={lam:<5.1f} {qa:>9.4f} {sur:>10.4f} {qa+sur:>9.4f}")

# Best linear merge (minimise total loss)
best_lam = min(linear_results, key=lambda l: sum(linear_results[l]))
best_qa, best_sum = linear_results[best_lam]
print(f"\nBest linear merge: λ={best_lam} → QA {best_qa:.4f}, Sum {best_sum:.4f}")
print(f"vs Adapter A:  QA {qa_a:.4f}, Sum {sum_a:.4f}")
print(f"vs Adapter B:  QA {qa_b:.4f}, Sum {sum_b:.4f}")

λ          QA loss   Sum loss     Total
----------------------------------------


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

λ=0.0      4.4827     2.8832    7.3659


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

λ=0.3      4.3686     2.9566    7.3253


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

λ=0.5      4.3042     3.0122    7.3164


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

λ=0.7      4.2515     3.0723    7.3238


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

λ=1.0      4.1931     3.1695    7.3626

Best linear merge: λ=0.5 → QA 4.3042, Sum 3.0122
vs Adapter A:  QA 4.1931, Sum 3.1695
vs Adapter B:  QA 4.4827, Sum 2.8832


## Advanced Methods: TIES, DARE, and SLERP

Linear interpolation has a flaw: if Task A pushes weight `w` to +0.1 and Task B pushes it to -0.1, averaging gives 0 — neither task's knowledge survives.

### TIES (Trim, Elect Sign, Merge)
Introduced in [Resolving Interference When Merging Models (2023)](https://arxiv.org/abs/2306.01708):

1. **Trim**: For each adapter, keep only the top `density`% of parameters by magnitude — zero out small, noisy deltas
2. **Elect Sign**: At each parameter position, elect the sign with the larger total magnitude across all adapters (majority vote by weight)
3. **Merge**: Average only the parameters that agree with the elected sign — ignore conflicting contributors

### DARE (Drop And REscale)
Introduced in [Language Models are Super Mario (2024)](https://arxiv.org/abs/2311.03099):

1. **Drop**: Randomly zero out a fraction `p` of each task vector's parameters
2. **REscale**: Multiply survivors by `1/(1-p)` to preserve expected magnitude
3. **Merge**: Linearly combine the sparsified, rescaled vectors

DARE leverages the observation that most delta parameters are redundant — 90%+ can be dropped with minimal quality impact.

### SLERP (Spherical Linear Interpolation)
Instead of linear interpolation in weight space, interpolate along the **great circle** on the hypersphere between the two delta vectors:

$$\text{SLERP}(\mathbf{a}, \mathbf{b}, t) = \frac{\sin((1-t)\theta)}{\sin\theta}\mathbf{a} + \frac{\sin(t\theta)}{\sin\theta}\mathbf{b}$$

where $\theta = \cos^{-1}\left(\frac{\mathbf{a} \cdot \mathbf{b}}{|\mathbf{a}||\mathbf{b}|}\right)$. Preserves the magnitude of intermediate vectors better than linear blending.

In [6]:
# ── TIES Merge ──
def ties_merge(da, db, density=0.6, lam=0.5):
    """Trim → Elect Sign → Merge (TIES algorithm)."""
    def trim(d, dens):
        """Keep top `dens` fraction by magnitude, zero out the rest."""
        if d.numel() <= 1:
            return d
        k = max(1, int(d.numel() * dens))
        threshold = torch.topk(d.abs().flatten(), k, largest=True).values[-1]
        return d * (d.abs() >= threshold)

    keys = set(da) | set(db)
    merged = {}
    for k in keys:
        if k not in da or k not in db:
            merged[k] = da.get(k, db.get(k))
            continue
        ta = trim(da[k].float(), density)
        tb = trim(db[k].float(), density)
        # Elect sign: where sum > 0 → +1; sum < 0 → -1; sum == 0 → +1
        elected = torch.sign(ta + tb)
        elected[elected == 0] = 1.0
        # Keep params agreeing with elected sign (or zero params)
        agree_a = ((torch.sign(ta) == elected) | (ta == 0)).float()
        agree_b = ((torch.sign(tb) == elected) | (tb == 0)).float()
        denom = (agree_a + agree_b).clamp(min=1.0)
        merged[k] = ((ta * agree_a + tb * agree_b) / denom).to(da[k].dtype)
    return merged

# ── DARE Merge ──
def dare_merge(da, db, drop_rate=0.3, lam=0.5, seed=42):
    """Drop And REscale, then linearly combine."""
    rng = torch.Generator()
    rng.manual_seed(seed)
    keys = set(da) | set(db)
    merged = {}
    for k in keys:
        if k not in da or k not in db:
            merged[k] = da.get(k, db.get(k))
            continue
        a = da[k].float()
        b = db[k].float()
        mask_a = (torch.rand(a.shape, generator=rng) > drop_rate).float()
        mask_b = (torch.rand(b.shape, generator=rng) > drop_rate).float()
        ra = a * mask_a / (1.0 - drop_rate + 1e-8)
        rb = b * mask_b / (1.0 - drop_rate + 1e-8)
        merged[k] = (lam * ra + (1 - lam) * rb).to(da[k].dtype)
    return merged

# ── SLERP Merge ──
def slerp_vectors(a, b, t=0.5, eps=1e-8):
    """Spherical linear interpolation between two tensors (as vectors)."""
    a_f, b_f = a.flatten().float(), b.flatten().float()
    a_n, b_n = torch.norm(a_f), torch.norm(b_f)
    if a_n < eps or b_n < eps:
        return ((1 - t) * a_f + t * b_f).reshape(a.shape).to(a.dtype)
    a_u, b_u = a_f / a_n, b_f / b_n
    dot = (a_u * b_u).sum().clamp(-1 + eps, 1 - eps)
    theta = torch.acos(dot.abs())
    if theta.abs() < eps:
        return ((1 - t) * a_f + t * b_f).reshape(a.shape).to(a.dtype)
    sin_t = torch.sin(theta)
    result = (torch.sin((1 - t) * theta) / sin_t * a_f +
              torch.sin(t * theta) / sin_t * b_f)
    return result.reshape(a.shape).to(a.dtype)

def slerp_merge(da, db, t=0.5):
    """SLERP merge of two delta dictionaries."""
    keys = set(da) | set(db)
    merged = {}
    for k in keys:
        if k not in da or k not in db:
            merged[k] = da.get(k, db.get(k))
            continue
        merged[k] = slerp_vectors(da[k], db[k], t=t)
    return merged

# ── Run all advanced methods ──
print("Running TIES, DARE, SLERP merges …\n")

ties_delta  = ties_merge(delta_a, delta_b, density=0.6, lam=0.5)
dare_delta  = dare_merge(delta_a, delta_b, drop_rate=0.3, lam=0.5)
slerp_delta = slerp_merge(delta_a, delta_b, t=0.5)

qa_ties,  sum_ties  = eval_from_delta(ties_delta,  "TIES")
qa_dare,  sum_dare  = eval_from_delta(dare_delta,  "DARE")
qa_slerp, sum_slerp = eval_from_delta(slerp_delta, "SLERP")

# Best linear for comparison
lin_delta = linear_merge(delta_a, delta_b, lam=best_lam)
qa_lin, sum_lin = linear_results[best_lam]

print(f"{'Method':<22} {'QA loss':>9} {'Sum loss':>10} {'Total':>9}")
print("-" * 52)
print(f"{'Linear (best λ)':<22} {qa_lin:>9.4f} {sum_lin:>10.4f} {qa_lin+sum_lin:>9.4f}")
print(f"{'TIES (d=0.6)':<22} {qa_ties:>9.4f} {sum_ties:>10.4f} {qa_ties+sum_ties:>9.4f}")
print(f"{'DARE (p=0.3)':<22} {qa_dare:>9.4f} {sum_dare:>10.4f} {qa_dare+sum_dare:>9.4f}")
print(f"{'SLERP (t=0.5)':<22} {qa_slerp:>9.4f} {sum_slerp:>10.4f} {qa_slerp+sum_slerp:>9.4f}")

Running TIES, DARE, SLERP merges …



Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Method                   QA loss   Sum loss     Total
----------------------------------------------------
Linear (best λ)           4.3042     3.0122    7.3164
TIES (d=0.6)              4.2718     2.9843    7.2561
DARE (p=0.3)              4.3088     3.0129    7.3217
SLERP (t=0.5)             4.2476     2.9532    7.2008


## 🔴 Failure Point: Merging Incompatible Adapters

Model merging assumes the adapters being merged are **compatible** — trained from the **same base model** and on **related tasks** with **similar magnitudes** of weight change.

### What Goes Wrong

| Incompatibility | Effect | Why |
|---|---|---|
| Different base models | Catastrophic collapse | Weight spaces are not aligned |
| Wildly different tasks (e.g., code vs. poetry) | Mutual interference | Conflicting gradient directions for all parameters |
| Very different adapter scales (r=4 vs r=64) | Dominant adapter overwhelms | One task vector dwarfs the other |
| **Random / adversarial adapter** | Both tasks degrade | Noise corrupts learned task direction |

### The Demo

We simulate the worst case: replace Adapter B with a **randomly initialized noise adapter** (same LoRA structure, random weights). Then merge with the genuine Adapter A.

**Expected outcome**: The merged model performs *worse than both standalone adapters* on both tasks — the noise adapter acts as gradient pollution, corrupting the clean task vector from Adapter A.

In [7]:
# ── Failure Demo: Random Noise Adapter ──
print("Building random noise adapter …")

# Create a random LoRA adapter with same structure as delta_a
# Scale noise to match the magnitude of delta_a
torch.manual_seed(99)
delta_noise = {}
for k, v in delta_a.items():
    # Random noise scaled to same magnitude as Adapter A's delta
    scale = v.abs().mean() * 2.0  # deliberately destructive scale
    delta_noise[k] = (torch.randn_like(v) * scale).to(v.dtype)

# Sanity check: compare magnitudes
da_mag = torch.cat([v.flatten() for v in delta_a.values()]).abs().mean().item()
dn_mag = torch.cat([v.flatten() for v in delta_noise.values()]).abs().mean().item()
print(f"  Adapter A delta magnitude (mean abs): {da_mag:.6f}")
print(f"  Noise  adapter magnitude (mean abs):  {dn_mag:.6f}")

# Merge: Adapter A (good) + Random Noise (bad)
fail_delta = linear_merge(delta_a, delta_noise, lam=0.5)
qa_fail, sum_fail = eval_from_delta(fail_delta, "A+Noise")

# Also show pure noise adapter for reference
qa_noise, sum_noise = eval_from_delta(delta_noise, "Noise only")

print(f"\n{'Model':<30} {'QA loss':>9} {'Sum loss':>10}")
print("-" * 51)
print(f"{'Adapter A (QA specialist)':<30} {qa_a:>9.4f} {sum_a:>10.4f}")
print(f"{'Merged: A + Noise (failure)':<30} {qa_fail:>9.4f} {sum_fail:>10.4f}")
print(f"{'Noise only (control)':<30} {qa_noise:>9.4f} {sum_noise:>10.4f}")
print(f"{'Best merge (TIES/DARE/linear)':<30} {min(qa_ties,qa_dare,qa_lin):>9.4f} {min(sum_ties,sum_dare,sum_lin):>10.4f}")
print(f"\n🔴 Failure: merging with noise adapter degrades QA from {qa_a:.4f} → {qa_fail:.4f}")

Building random noise adapter …
  Adapter A delta magnitude (mean abs): 0.000170
  Noise  adapter magnitude (mean abs):  0.000272


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


Model                            QA loss   Sum loss
---------------------------------------------------
Adapter A (QA specialist)         4.1931     3.1695
Merged: A + Noise (failure)       4.3141     3.1719
Noise only (control)              4.5083     3.1779
Best merge (TIES/DARE/linear)     4.2718     2.9843

🔴 Failure: merging with noise adapter degrades QA from 4.1931 → 4.3141


## Results & Key Takeaways

### What We Observed

| Observation | Implication |
|---|---|
| Adapter A: great QA, poor Sum | Task specialization is real — LoRA adapters focus on their training distribution |
| Adapter B: poor QA, great Sum | The same architecture, same base, very different task vectors |
| Linear merge (λ=0.5): decent on both | Weight arithmetic works! Multi-task capability at zero extra training cost |
| TIES/DARE: equal or better than linear | Resolving sign conflicts and dropping noise improves quality |
| SLERP: smooth interpolation | Preserves vector magnitude better, useful when adapter scales differ |
| Noise merge: worse than all valid merges | **Compatibility is a prerequisite** — merging requires task alignment |

### When to Use Which Method

- **Linear**: Simple, fast, interpretable. Good starting point. Use λ based on task priority.
- **TIES**: Best when adapters have conflicting update directions (verified on many benchmarks).
- **DARE**: When you expect high redundancy in delta parameters (large adapters, many tasks).
- **SLERP**: When adapter scales differ significantly or you want smooth semantic interpolation.

### Practical Checklist

✅ Same base model for all adapters  
✅ Similar training data quality and size  
✅ Validate merged model on **all tasks** before deployment  
✅ Start with λ=0.5 linear, upgrade to TIES if quality is insufficient  
⚠️ Never merge adapters from different base model families  
⚠️ Evaluate failure modes: quality collapse is silent without per-task evaluation

In [8]:
# ── Full Summary Table ──
print("=" * 72)
print("  MODEL MERGING — COMPLETE RESULTS SUMMARY")
print("=" * 72)
print(f"  Model : {MODEL}")
print(f"  Tasks : Q&A (16 pairs) + Summarization (16 pairs)")
print(f"  Base  : {MODEL.split('/')[-1]}")
print("=" * 72)
print(f"{'Model / Method':<32} {'QA val loss':>12} {'Sum val loss':>13} {'Avg':>8}")
print("-" * 72)

rows = [
    ("Base (no fine-tuning)",     qa_base,  sum_base,  "—"),
    ("Adapter A  [QA only]",      qa_a,     sum_a,     "A specialist"),
    ("Adapter B  [Sum only]",     qa_b,     sum_b,     "B specialist"),
    ("─── Merging (λ=0.0=pure B)", qa_b,    sum_b,     ""),
    ("Linear λ=0.3 (A-dominant)", *linear_results[0.3], ""),
    ("Linear λ=0.5 (balanced)",   *linear_results[0.5], ""),
    ("Linear λ=0.7 (B-dominant)", *linear_results[0.7], ""),
    ("─── Advanced Methods",       None,     None,      ""),
    ("TIES  (density=0.6)",       qa_ties,  sum_ties,  ""),
    ("DARE  (drop=0.3, λ=0.5)",   qa_dare,  sum_dare,  ""),
    ("SLERP (t=0.5)",             qa_slerp, sum_slerp, ""),
    ("─── Failure Demo",           None,     None,      ""),
    ("Merge: A + Noise adapter",  qa_fail,  sum_fail,  "🔴 incompatible"),
]

for label, qa, sur, note in rows:
    if qa is None:
        print(f"  {label}")
    else:
        avg = (qa + sur) / 2
        flag = " ←" if avg == min((r[1]+r[2])/2 for r in rows if r[1] is not None and r[0] not in ("Base (no fine-tuning)", "Merge: A + Noise adapter")) else ""
        note_str = f"  [{note}]" if note else ""
        print(f"  {label:<30} {qa:>12.4f} {sur:>13.4f} {avg:>8.4f}{flag}{note_str}")

print("=" * 72)
print(f"  Lower val loss = better. Both tasks evaluated simultaneously.")

  MODEL MERGING — COMPLETE RESULTS SUMMARY
  Model : HuggingFaceTB/SmolLM2-135M-Instruct
  Tasks : Q&A (16 pairs) + Summarization (16 pairs)
  Base  : SmolLM2-135M-Instruct
Model / Method                    QA val loss  Sum val loss      Avg
------------------------------------------------------------------------
  Base (no fine-tuning)                4.5077        3.1772   3.8425  [—]
  Adapter A  [QA only]                 4.1931        3.1695   3.6813  [A specialist]
  Adapter B  [Sum only]                4.4827        2.8832   3.6830  [B specialist]
  ─── Merging (λ=0.0=pure B)           4.4827        2.8832   3.6830
  Linear λ=0.3 (A-dominant)            4.3686        2.9566   3.6626
  Linear λ=0.5 (balanced)              4.3042        3.0122   3.6582
  Linear λ=0.7 (B-dominant)            4.2515        3.0723   3.6619
  ─── Advanced Methods
  TIES  (density=0.6)                  4.2718        2.9843   3.6280
  DARE  (drop=0.3, λ=0.5)              4.3088        3.0129   3.6609
  SL

In [9]:
# ── 3-Panel Comparison Chart ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Model Merging Comparison — SmolLM2-135M", fontsize=14, fontweight="bold", y=1.01)

COLORS = {
    "Base":      "#aaaaaa",
    "A only":    "#4c8bf5",
    "B only":    "#f5a623",
    "Lin 0.5":   "#34a853",
    "TIES":      "#0b8c4b",
    "DARE":      "#1a73e8",
    "SLERP":     "#8e44ad",
    "Failure":   "#ea4335",
}

# ── Panel 1: Specialist vs Merged (QA loss + Sum loss) ──
ax = axes[0]
labels = ["Base", "A only\n(QA)", "B only\n(Sum)", "Linear\nλ=0.5", "TIES", "DARE", "SLERP"]
qa_vals  = [qa_base, qa_a, qa_b, linear_results[0.5][0], qa_ties, qa_dare, qa_slerp]
sum_vals = [sum_base, sum_a, sum_b, linear_results[0.5][1], sum_ties, sum_dare, sum_slerp]

x = np.arange(len(labels))
w = 0.38
bars_qa  = ax.bar(x - w/2, qa_vals,  w, label="QA loss",  color="#4c8bf5", alpha=0.85, edgecolor="white")
bars_sum = ax.bar(x + w/2, sum_vals, w, label="Sum loss", color="#f5a623", alpha=0.85, edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8.5)
ax.set_ylabel("Validation Loss"); ax.set_title("Specialist vs Merged Models", fontweight="bold")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for bar in list(bars_qa) + list(bars_sum):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)

# ── Panel 2: λ sweep — QA vs Sum tradeoff ──
ax = axes[1]
lam_vals = list(lambdas)
qa_sweep  = [linear_results[l][0] for l in lam_vals]
sum_sweep = [linear_results[l][1] for l in lam_vals]
ax.plot(lam_vals, qa_sweep,  "o-", color="#4c8bf5", label="QA loss",  linewidth=2, markersize=7)
ax.plot(lam_vals, sum_sweep, "s-", color="#f5a623", label="Sum loss", linewidth=2, markersize=7)
ax.axvline(x=best_lam, color="#34a853", linestyle="--", alpha=0.7, label=f"Best λ={best_lam}")
ax.fill_between(lam_vals, qa_sweep, sum_sweep, alpha=0.08, color="gray")
ax.set_xlabel("λ  (0=pure B, 1=pure A)"); ax.set_ylabel("Validation Loss")
ax.set_title("Linear Merge λ Sweep\n(QA vs Sum tradeoff)", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
for x_v, q, s in zip(lam_vals, qa_sweep, sum_sweep):
    ax.annotate(f"{q:.2f}", (x_v, q), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=7, color="#4c8bf5")
    ax.annotate(f"{s:.2f}", (x_v, s), textcoords="offset points", xytext=(0, -12), ha="center", fontsize=7, color="#f5a623")

# ── Panel 3: Method comparison — total loss + failure ──
ax = axes[2]
methods  = ["Adapter A\n(QA)", "Adapter B\n(Sum)", "Linear\nλ=0.5", "TIES", "DARE", "SLERP", "Failure\n(A+Noise)"]
totals   = [
    qa_a + sum_a,
    qa_b + sum_b,
    linear_results[0.5][0] + linear_results[0.5][1],
    qa_ties  + sum_ties,
    qa_dare  + sum_dare,
    qa_slerp + sum_slerp,
    qa_fail  + sum_fail,
]
bar_colors = ["#4c8bf5","#f5a623","#34a853","#0b8c4b","#1a73e8","#8e44ad","#ea4335"]
bars = ax.bar(methods, totals, color=bar_colors, alpha=0.85, edgecolor="white")
ax.set_ylabel("Total Val Loss (QA + Sum)"); ax.set_title("Merging Method Comparison\n(lower = better multi-task)", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, totals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticklabels(methods, fontsize=8)
# Highlight failure bar
bars[-1].set_hatch("//")
bars[-1].set_edgecolor("#b71c1c")

plt.tight_layout()
out_path = "merging_comparison.png"
plt.savefig(out_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Chart saved to {out_path}")
free_memory()

/var/folders/1v/sr7z49p93fd794zxx2z8b9xr0000gp/T/ipykernel_88535/399566330.py:68: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(methods, fontsize=8)


Chart saved to merging_comparison.png


/var/folders/1v/sr7z49p93fd794zxx2z8b9xr0000gp/T/ipykernel_88535/399566330.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

| Concept | What We Learned |
|---|---|
| **Task vectors** | Fine-tuned models = base model + small weight delta. These deltas carry task knowledge. |
| **Linear merge** | Arithmetic on task vectors gives a multi-task model. λ controls task priority tradeoff. |
| **TIES** | Trim noise, resolve sign conflicts before merging → handles parameter interference. |
| **DARE** | Drop 30% of delta parameters, rescale survivors → sparse merging with less interference. |
| **SLERP** | Interpolate on hypersphere → better magnitude preservation for differently-scaled adapters. |
| **Failure** | Noise adapter corrupts clean task vectors. Compatibility check is non-negotiable. |

### Next Up → Step 14: Responsible & Safe Fine-Tuning (`12_responsible_safe_finetuning.ipynb`)

Fine-tuning can silently **erode safety guardrails**. Even training on benign data can reduce a model's refusal rate by 20–40%. Next notebook covers risk taxonomy, measuring guardrail retention, and mitigation strategies including data curation and DPO safety alignment.